# Tutorial 04: Multi-Factor Selection and Combination

Learn how to combine multiple factors, manage redundancy, and build composite signals.

## What You'll Learn
- Calculate factor correlation and redundancy
- Factor orthogonalization techniques
- Equal-weight and IC-weighted combinations
- Incremental IC analysis
- Factor portfolio construction

In [ ]:
import sys
import numpy as np
import pandas as pd
from typing import List, Dict

sys.path.insert(0, '/home/shw/quant_projects/factor_engine')
sys.path.insert(0, '/home/shw/quant_projects/notebooks')

from utils import (
    display_success, display_warning, display_metrics,
    NotebookTimer, ProgressBar, display_dataframe_summary
)

## Step 1: Generate Multiple Factors

Create a diverse set of factors across different categories.

In [ ]:
from api import col, rank, ts_mean, ts_std, delay, zscore, ts_corr, Factor
from backend.pandas_backend import PandasBackend
from storage.datasource import DataSource
from runtime.engine import FactorEngine

# Generate synthetic data
def generate_market_data(n_stocks=100, n_days=400):
    np.random.seed(42)
    dates = pd.date_range(end='2024-01-01', periods=n_days, freq='B')
    tickers = [f'STOCK_{i:03d}' for i in range(n_stocks)]
    
    data = []
    for ticker in tickers:
        base = 50 + np.random.randn() * 20
        prices = base * np.exp(np.cumsum(np.random.randn(n_days) * 0.02))
        volume = np.abs(np.random.randn(n_days) * 1e6 + 5e6)
        
        for i, date in enumerate(dates):
            data.append({
                'date': date,
                'ticker': ticker,
                'close': prices[i],
                'open': prices[i] * (1 + np.random.randn() * 0.005),
                'high': prices[i] * (1 + abs(np.random.randn() * 0.01)),
                'low': prices[i] * (1 - abs(np.random.randn() * 0.01)),
                'volume': volume[i],
            })
    
    df = pd.DataFrame(data).set_index(['date', 'ticker']).sort_index()
    df = df.join(df.groupby(level='ticker')['close'].pct_change(5).shift(-5).rename('fwd_return_5d'))
    return df

market_data = generate_market_data()

class PandasDataSource(DataSource):
    def __init__(self, df):
        self.df = df
    def load_column(self, name: str):
        return self.df[name]

data_source = PandasDataSource(market_data)
engine = FactorEngine(backend=PandasBackend(), data_source=data_source)

display_success("Data and engine ready")

## Step 2: Define Factor Library

Create factors from different categories: momentum, volatility, value, volume.

In [ ]:
# Define factor library
factor_definitions = [
    # Momentum factors
    ('momentum_20_5', rank(ts_mean(col('close'), 20) - ts_mean(col('close'), 5))),
    ('momentum_60_20', rank(ts_mean(col('close'), 60) - ts_mean(col('close'), 20))),
    ('momentum_120', rank(col('close') / delay(col('close'), 120) - 1)),
    
    # Volatility factors
    ('volatility_20', rank(-ts_std(col('close') / delay(col('close'), 1) - 1, 20))),
    ('volatility_60', rank(-ts_std(col('close') / delay(col('close'), 1) - 1, 60))),
    
    # Mean reversion
    ('reversion_5', rank(-(col('close') / delay(col('close'), 5) - 1))),
    ('zscore_reversion', rank(-zscore(col('close') / delay(col('close'), 1) - 1))),
    
    # Volume factors
    ('volume_trend', rank(ts_mean(col('volume'), 5) / ts_mean(col('volume'), 20))),
    ('volume_momentum', rank(col('volume') / delay(col('volume'), 20))),
]

print(f"Defined {len(factor_definitions)} factors")
for name, _ in factor_definitions:
    print(f"  - {name}")

## Step 3: Calculate All Factors

Run all factors and collect results.

In [ ]:
factor_results = {}
progress = ProgressBar(len(factor_definitions), "Calculating factors")

for name, expr in factor_definitions:
    factor = Factor(name, expr, '1d', 'equities')
    try:
        result = engine.run(factor)
        factor_results[name] = result['result']
    except Exception as e:
        print(f"Failed {name}: {e}")
    progress.update(1)

progress.close()

# Combine into DataFrame
factor_df = pd.DataFrame(factor_results)

print(f"\nCalculated {len(factor_df.columns)} factors")
print(f"Shape: {factor_df.shape}")
print(f"Coverage: {factor_df.notna().sum().mean() / len(factor_df) * 100:.1f}%")

display_dataframe_summary(factor_df, "Factor Universe")

## Step 4: Factor Correlation Analysis

Identify redundant factors through correlation matrix.

In [ ]:
# Calculate correlation matrix
corr_matrix = factor_df.corr(method='spearman')

print("Factor Correlation Matrix:")
print(corr_matrix.round(3).to_string())

# Find highly correlated pairs
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) > 0.7:
            high_corr_pairs.append({
                'factor1': corr_matrix.columns[i],
                'factor2': corr_matrix.columns[j],
                'correlation': corr_val
            })

if high_corr_pairs:
    print(f"\nFound {len(high_corr_pairs)} highly correlated pairs (|corr| > 0.7):")
    for pair in high_corr_pairs:
        print(f"  {pair['factor1']} <-> {pair['factor2']}: {pair['correlation']:.3f}")
    display_warning("High correlation detected - consider removing redundant factors")
else:
    display_success("No highly correlated factor pairs found")

## Step 5: Individual Factor Performance

Evaluate each factor's predictive power.

In [ ]:
def calculate_factor_ic(factor_values: pd.Series, forward_returns: pd.Series) -> Dict[str, float]:
    """Calculate IC metrics for a single factor."""
    aligned = pd.DataFrame({
        'factor': factor_values,
        'returns': forward_returns
    }).dropna()
    
    if len(aligned) < 10:
        return {'ic': np.nan, 'ic_mean': np.nan, 'ic_std': np.nan, 'ic_ir': np.nan}
    
    # Overall IC
    overall_ic = aligned['factor'].corr(aligned['returns'], method='spearman')
    
    # Time-series IC
    dates = aligned.index.get_level_values(0).unique()
    ic_series = []
    for date in dates:
        try:
            cross = aligned.xs(date, level=0)
            if len(cross) >= 10:
                ic = cross['factor'].corr(cross['returns'], method='spearman')
                ic_series.append(ic)
        except:
            pass
    
    ic_mean = np.mean(ic_series) if ic_series else np.nan
    ic_std = np.std(ic_series) if ic_series else np.nan
    ic_ir = ic_mean / ic_std if ic_std > 0 else np.nan
    
    return {
        'ic': overall_ic,
        'ic_mean': ic_mean,
        'ic_std': ic_std,
        'ic_ir': ic_ir
    }

# Evaluate all factors
forward_returns = market_data['fwd_return_5d']
performance = []

for factor_name in factor_df.columns:
    metrics = calculate_factor_ic(factor_df[factor_name], forward_returns)
    metrics['factor'] = factor_name
    performance.append(metrics)

perf_df = pd.DataFrame(performance).sort_values('ic_ir', ascending=False)

print("\nFactor Performance Ranking:")
print(perf_df.to_string(index=False))

# Highlight top factors
top_3 = perf_df.head(3)
print("\nTop 3 factors by IC IR:")
for _, row in top_3.iterrows():
    print(f"  {row['factor']}: IC={row['ic']:.4f}, IC_IR={row['ic_ir']:.4f}")

## Step 6: Factor Combination Strategies

Combine factors using different weighting schemes.

In [ ]:
def combine_factors_equal_weight(factor_df: pd.DataFrame, factor_names: List[str]) -> pd.Series:
    """Equal-weight factor combination."""
    subset = factor_df[factor_names].copy()
    # Standardize each factor
    for col in subset.columns:
        subset[col] = (subset[col] - subset[col].mean()) / subset[col].std()
    return subset.mean(axis=1)

def combine_factors_ic_weight(factor_df: pd.DataFrame, factor_names: List[str], 
                               ics: Dict[str, float]) -> pd.Series:
    """IC-weighted factor combination."""
    subset = factor_df[factor_names].copy()
    
    # Standardize and weight by IC
    weighted_sum = 0
    total_weight = 0
    
    for col in subset.columns:
        ic = ics.get(col, 0)
        if ic > 0:
            standardized = (subset[col] - subset[col].mean()) / subset[col].std()
            weighted_sum += standardized * ic
            total_weight += ic
    
    return weighted_sum / total_weight if total_weight > 0 else weighted_sum

# Select top N factors
top_n = 5
selected_factors = perf_df.head(top_n)['factor'].tolist()
ic_weights = {row['factor']: row['ic'] for _, row in perf_df.iterrows()}

print(f"\nSelected top {top_n} factors:")
for f in selected_factors:
    print(f"  - {f} (IC: {ic_weights[f]:.4f})")

# Create combinations
combo_equal = combine_factors_equal_weight(factor_df, selected_factors)
combo_ic = combine_factors_ic_weight(factor_df, selected_factors, ic_weights)

# Evaluate combinations
equal_perf = calculate_factor_ic(combo_equal, forward_returns)
ic_perf = calculate_factor_ic(combo_ic, forward_returns)

print("\nCombination Performance:")
print(f"\nEqual-weight:")
for k, v in equal_perf.items():
    print(f"  {k}: {v:.4f}")

print(f"\nIC-weighted:")
for k, v in ic_perf.items():
    print(f"  {k}: {v:.4f}")

# Compare to best single factor
best_single = perf_df.iloc[0]
print(f"\nBest single factor ({best_single['factor']}): IC_IR={best_single['ic_ir']:.4f}")
print(f"Equal-weight combo: IC_IR={equal_perf['ic_ir']:.4f}")
print(f"IC-weighted combo: IC_IR={ic_perf['ic_ir']:.4f}")

if ic_perf['ic_ir'] > best_single['ic_ir']:
    improvement = (ic_perf['ic_ir'] - best_single['ic_ir']) / best_single['ic_ir'] * 100
    display_success(f"Combination improved IC_IR by {improvement:.1f}%")
else:
    display_warning("Combination did not outperform best single factor")

## Step 7: Incremental IC Analysis

Measure the incremental contribution of each factor.

In [ ]:
def incremental_ic_analysis(factor_df: pd.DataFrame, factor_order: List[str], 
                            forward_returns: pd.Series):
    """Calculate incremental IC when adding factors sequentially."""
    results = []
    
    for i in range(1, len(factor_order) + 1):
        factors_used = factor_order[:i]
        combo = combine_factors_equal_weight(factor_df, factors_used)
        perf = calculate_factor_ic(combo, forward_returns)
        
        incremental = perf['ic'] - results[-1]['cumulative_ic'] if results else perf['ic']
        
        results.append({
            'n_factors': i,
            'added_factor': factors_used[-1],
            'cumulative_ic': perf['ic'],
            'cumulative_ic_ir': perf['ic_ir'],
            'incremental_ic': incremental,
        })
    
    return pd.DataFrame(results)

# Analyze incremental contribution
incremental_df = incremental_ic_analysis(factor_df, selected_factors, forward_returns)

print("\nIncremental IC Analysis:")
print(incremental_df.to_string(index=False))

# Find optimal number of factors
optimal_idx = incremental_df['cumulative_ic_ir'].idxmax()
optimal_n = incremental_df.loc[optimal_idx, 'n_factors']

display_metrics({
    'optimal_n_factors': optimal_n,
    'optimal_ic': incremental_df.loc[optimal_idx, 'cumulative_ic'],
    'optimal_ic_ir': incremental_df.loc[optimal_idx, 'cumulative_ic_ir'],
}, "Optimal Factor Count")

print(f"\nOptimal portfolio uses {int(optimal_n)} factors")

## Step 8: Final Composite Factor

Build the final factor using optimal selection and weighting.

In [ ]:
# Use optimal number of factors
final_factors = selected_factors[:int(optimal_n)]
final_composite = combine_factors_ic_weight(factor_df, final_factors, ic_weights)

# Final evaluation
final_perf = calculate_factor_ic(final_composite, forward_returns)

print(f"\nFinal Composite Factor:")
print(f"  Constituents: {', '.join(final_factors)}")
print(f"\nPerformance:")
for k, v in final_perf.items():
    print(f"  {k}: {v:.4f}")

display_success("Final composite factor constructed")

# Export factor for use
print(f"\nFactor shape: {final_composite.shape}")
print(f"Coverage: {final_composite.notna().sum() / len(final_composite) * 100:.1f}%")

## Key Takeaways

1. **Diversification**: Combine factors from different categories (momentum, volatility, value)
2. **Correlation Check**: Remove highly correlated factors to avoid redundancy
3. **Individual Performance**: Always evaluate factors independently first
4. **Weighting**: IC-weighting typically outperforms equal-weighting
5. **Incremental Analysis**: More factors isn't always better - find the optimal count
6. **Orthogonalization**: Consider orthogonalizing factors to remove shared components

## Best Practices

- Start with 20-30 candidate factors across multiple categories
- Remove factors with |correlation| > 0.7 unless they have distinct signals
- Use walk-forward validation for combination weights
- Rebalance combination weights periodically (quarterly/annually)
- Monitor factor contributions over time - some may decay

## Next Steps

- Tutorial 05: Full workflow from research to production
- Examples: Real-world multi-factor strategies